![DB Academy](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/common/db-academy.png)

# 11L - Use a Databricks Default Declarative Automation Bundle (DAB) Template

### Estimated Duration: ~10 minutes

## Overview

Up to this point you've authored bundles by hand. In this lab, you'll use the Databricks CLI's built-in **bundle templates** to scaffold a new project from scratch. Templates give you a working bundle with sensible defaults (job, pipeline, tests, README) that you can customize, instead of starting from a blank **databricks.yml**.

You'll inspect the available templates, generate a `default-python` project with `databricks bundle init`, explore the generated structure, and validate it.

## Learning Objectives

By the end of this lab, you will be able to:

1. **List the default bundle templates** Databricks ships with the CLI.
2. **Initialize a new bundle** from a template using `databricks bundle init <template>`.
3. **Navigate the generated project structure** (`resources/`, `src/`, `scratch/`, `tests/`, `databricks.yml`).
4. **Validate the generated bundle** with `databricks bundle validate`, including the working-directory pitfall when the bundle lives in a subfolder.

## Reference Documentation

- **What are bundles?** (intro): [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/)
- **Bundle templates (default and custom)**: [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/templates) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/templates) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/templates)
- **`databricks bundle` CLI commands** (includes `bundle init`): [AWS](https://docs.databricks.com/aws/en/dev-tools/cli/bundle-commands) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/cli/bundle-commands) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/cli/bundle-commands)
- **Bundle configuration reference**: [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/reference) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/reference) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/reference)
- **MLOps Stacks** (advanced template): [AWS](https://docs.databricks.com/aws/en/machine-learning/mlops/mlops-stacks) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/mlops/mlops-stacks) | [GCP](https://docs.databricks.com/gcp/en/machine-learning/mlops/mlops-stacks)

## REQUIRED - SELECT A COMPUTE ENVIRONMENT
<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select All-Purpose Compute</strong>
  <div style="color:#333;">

This notebook requires **all-purpose compute** (Dedicated). Serverless is not supported for this notebook.

Follow these steps to attach an all-purpose compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your `labuser_USERNAME` cluster.
    - By default, the notebook might use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

⚠️ **NOTE:** If the cluster shows a **terminated** state (red dot in the cluster picker), it needs to be started before you can attach. Click the cluster, then **Start**, and wait a few minutes until you see a green dot.
  </div>
</div>


## A. Classroom Setup

Run the following cell to configure your working environment for this course.

**NOTE:** The `DA` object is only used in Databricks Academy courses and is not available outside of them. It dynamically references the information needed to run the course.

In [0]:
%run ../Includes/Classroom-Setup-11L

## B. Lab Scenario

You're starting a brand-new project and want to skip the boilerplate. Instead of writing **databricks.yml** by hand, you'll use the Databricks CLI's `bundle init` command with the `default-python` template to scaffold a working bundle, then validate it.

## C. Pre-flight Checks

Before starting the lab tasks, run a couple of quick checks to confirm the Databricks CLI is installed and authenticated against your workspace.

### C1. Check the Databricks CLI Version

Run a CLI command to confirm the Databricks CLI version is **v0.298.0**.

In [0]:
%sh
databricks -v

### C2. Confirm CLI Authentication

Run the cell below to confirm the Databricks CLI is authenticated against your workspace. If authentication is broken, the cell will return an error rather than a list of catalogs.

In [0]:
%sh
databricks catalogs list

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    DATABRICKS CLI ERROR TROUBLESHOOTING:
  </strong>
  <div style="color:#333;">

  - If you encounter a Databricks CLI authentication error, it means the authentication was not successful. Confirm you ran the notebook using your **all purpose compute**.

  - If you encounter the error below, it means your **databricks.yml** file has syntax issues due to a modification. Even for non-DAB CLI commands, the **databricks.yml** file is still required, as it may contain important authentication details, such as the host and profile, which are utilized by the CLI commands.

![CLI Invalid YAML](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/databricks_cli_error_invalid_yaml.png)
  </div>
</div>



## D. Task 1 - Explore Available Bundle Templates

Databricks ships several default bundle templates with the CLI. Each one creates a working bundle for a specific use case so you don't have to start from a blank **databricks.yml**.

| Template | Description |
|----------|-------------|
| `default-python` | A bundle with a Python job and a Apache Spark™ Declarative Pipeline. Good starting point for most ETL projects. |
| `default-sql` | A bundle that defines a job which runs SQL queries on a SQL warehouse. |
| `dbt-sql` | A bundle that uses dbt-core for local development and a bundle for deployment. Includes a job with a dbt task and dbt profile configuration. |
| `mlops-stacks` | An advanced full-stack template for MLOps projects (model training, serving, monitoring). |

Templates evolve over time, so for the current list and what each one generates, see the **Bundle templates** documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/templates) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/templates) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/templates)

## E. Task 2 - View `bundle init` Documentation

Run the `--help` command to view documentation for the `databricks bundle init` command. Read the available flags and arguments before generating a project.

In [0]:
%sh
databricks bundle init --help

## F. Task 3 - Initialize a Bundle from `default-python`

Use `databricks bundle init <template-name>` to scaffold a new project. The cell below uses the `default-python` template.

**NOTE:** If you don't specify a template name, `bundle init` enters **interactive** mode and prompts you to choose. Interactive mode is not available when running `%sh` cells in a notebook, so we always specify the template name in this lab.

In [0]:
%sh
databricks bundle init default-python

## G. Task 4 - Explore the Generated Bundle Structure

Navigate to the generated **my_project** folder in the workspace file browser and inspect what `bundle init` produced.

### Step 4.1
 - Top-level folders

- **resources/** contains additional YAML files included by the bundle:
    - **my_project.job.yml**
    - **my_project.pipeline.yml**
- **scratch/** contains an exploration notebook for ad-hoc analysis.
- **src/** contains the production code (a Python notebook and a Spark Declarative Pipeline notebook).
- **tests/** contains unit and integration tests for the project.
- The project root also contains a **pytest.ini**, **README.md**, and a few config files.

### Step 4.2 - Open the generated **databricks.yml**

Open the **databricks.yml** file at the root of **my_project** and notice the template provides two `targets` (typically `dev` and `prod`).

## H. Task 5 - Validate the Generated Bundle

Now validate the generated bundle. There's a small wrinkle: `databricks bundle` commands always run against the **current working directory**, and the bundle was created in a subfolder (`my_project/`) of where the CLI was invoked. So you'll need to `cd` into `my_project/` first.

### Step 5.1 - Confirm your working directory

Run the cell below to confirm you are **not** yet inside `my_project/`.

In [0]:
%sh
pwd
ls

### Step 5.2 - Change directory and validate

In a single `%sh` cell:

1. `cd` into `my_project`.
2. Run `databricks bundle validate`.

**Why both in one cell?** Each `%sh` cell starts a fresh shell, so a `cd` in one cell does not persist into the next.

In [0]:
%sh
cd 'my_project'
databricks bundle validate

## I. Wrap-Up

**Note:** After validation, take a few minutes to skim the generated template, the **databricks.yml**, the resource YAMLs in `resources/`, and the source code in `src/`.

**Why we stop at validate in this lab:** the Databricks Academy lab environment restricts your ability to create clusters, so the generated bundle's job and pipeline cannot actually be deployed and run here. The end-to-end deploy flow has been covered in earlier modules.


**Want to use templates locally?** This lab also provides a working VS Code environment. To deploy a template using VS Code, see **08 - Using VS Code with Databricks**.

## Conclusion

In this lab you used a Databricks default bundle template to scaffold a new project end-to-end without writing **databricks.yml** by hand:

1. Reviewed the four default templates (`default-python`, `default-sql`, `dbt-sql`, `mlops-stacks`).
2. Read the `databricks bundle init --help` output to understand the available flags.
3. Generated a working bundle with `databricks bundle init default-python`.
4. Explored the resulting project structure (`resources/`, `src/`, `scratch/`, `tests/`, **databricks.yml**).
5. Validated the generated bundle with `cd my_project && databricks bundle validate`.

Templates are usually how you'll start a new project in real life. From here, the same `validate` / `deploy` / `run` / `destroy` workflow you've been practicing applies.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>